# β stabilisation and mode type: GFTM (FILTER=0.5) and TGLF against GS2 (M1_A1_NE, R3, R4)

γ vs β at fixed ky, one figure per code against GS2, for each case. Codes: GFTM widths 0.4 / 0.6 / 0.7 / 0.8
(`scan_beta_f05`, FILTER=0.5), TGLF default / V5 (`scan_beta_modes`) and TGLF V3 (older `scan_beta` scan). All scans
use GS2's own (ky, β) grid; TGLF V3's M1_A1_NE scan has two extra ky (0.2, 0.3) that are dropped. The single-ky
figures draw all three TGLF settings next to each code.

**Line styles.** GS2 black (dominant mode only). Code: dominant mode (`argmax` of `growth_rate` over `mode`, never
index 0) solid, the other mode dashed (`NMODES=2`).

**Markers = mode type** from `general_analysis.mode_classification` (cutoffs `T_BAL`, `T_TEAR`, see that module):
▲ MTM (tearing parity), ■ KBM (ballooning parity), ● mixed. Markers are drawn only where γ > 0.
- **GS2** uses all four indicators (tearing parameter T, ω sign, χe/χi, De/χe) from each run's final time.
- **GFTM / TGLF** linear runs have no per-mode flux weights, so they are classified on **T and ω only**, with the
  same cutoffs: KBM = ω>0 and T<`T_BAL`; MTM = ω<0 and T>`T_TEAR`; otherwise mixed. A code marker is therefore a
  looser test than a GS2 one, and "KBM" there means only "ballooning parity, ion direction".
- T for GFTM / TGLF is the `tearing_parameter` stored per mode in each leaf's `pyroscan.nc`. TGLF V3's older scan
  stores none, so its T is computed by `FieldLine` from the `A_par` eigenfunction in its `pyroscan.nc` (`eig_T`).
- Every scan is read with `PyroScan(pyroscan_json=..., load_base_pyro=True).load_gk_output(netcdf_file=...)`;
  no per-run outputs are loaded, except GS2's γ(t) in section (4), which a `pyroscan.nc` does not store.

**β axis.** Code scans store pyrokinetics β (= GS2's number × a per-case factor f); the x axis here is **GS2's β
number**, matched by index. f is printed below and checked constant. TGLF V3 stores GS2's number directly (f = 1),
but its `input.tglf` BETAE equals TGLF V5's at the same index (checked R4 and M1_A1_NE, lowest and highest β), so
the index match is the same physical β.

## Imports and settings

In [1]:
from pathlib import Path
import os
import numpy as np
import xarray as xr
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pyrokinetics import Pyro, PyroScan
from pyrokinetics.diagnostics.field_line import FieldLine
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "beta_stabilisation_f05"
data_root = Path(os.environ["GYRO_DATA_OUTPUT"]).expanduser()
run_template = "Runs"

cases = ["M1_A1_NE", "R3", "R4"]
reference_beta = {"M1_A1_NE": 0.036, "R3": 0.094, "R4": 0.053}   # same beta convention as the plot axis
ky_targets = [0.1, 0.2, 0.3, 0.5]          # snapped to the nearest GS2 grid value, shown in the panel titles

# GS2 reference: data_root / code / run_template / project / case / scan_information
gs2_code, gs2_project, gs2_scan = "GS2", "scan_beta", "parameter_scan_GS2"
# Codes, one figure each: label -> (code, project, scan_information = leaf). Same three path parts for every case.
codes = {
    "GFTM w0.4": ("GFTM", "scan_beta_f05", "w0p4"),
    "GFTM w0.6": ("GFTM", "scan_beta_f05", "w0p6"),
    "GFTM w0.7": ("GFTM", "scan_beta_f05", "w0p7"),
    "GFTM w0.8": ("GFTM", "scan_beta_f05", "w0p8"),
    "TGLF default": ("TGLF", "scan_beta_modes", "default"),
    "TGLF V5": ("TGLF", "scan_beta_modes", "v5"),
    "TGLF V3": ("TGLF", "scan_beta", "parameter_scan_tglf_v3"),   # older scan: no tearing_parameter, beta stored as GS2's number
}
metadata_file, output_file = "pyroscan.json", "pyroscan.nc"   # in every scan directory
gamma_floor = 0.0                                        # markers only where gamma > this
n_spot, seed = 3, 0                                      # spot-check cells per (case, code family)

## Load data

Every leaf as a `PyroScan` from its `pyroscan.json` + `pyroscan.nc` (base pyro from `pyroscan_base.input`); the
plain datasets below are its `gk_output` with units stripped. GS2 and the codes must share the ky grid exactly; nothing is
interpolated.

In [2]:
def scan(code, project, case, leaf):
    d = data_root / code / run_template / project / case / leaf
    s = PyroScan(pyroscan_json=d / metadata_file, load_base_pyro=True)
    s.load_gk_output(netcdf_file=d / output_file)
    return s

gs2_pyroscan = {c: scan(gs2_code, gs2_project, c, gs2_scan) for c in cases}
model_pyroscan = {(c, lab): scan(code, project, c, leaf) for c in cases for lab, (code, project, leaf) in codes.items()}
gs2 = {c: s.gk_output.data.pint.dequantify() for c, s in gs2_pyroscan.items()}
model = {k: s.gk_output.data.pint.dequantify() for k, s in model_pyroscan.items()}

for (c, lab), ds in model.items():   # exact selection on GS2's ky (KeyError if one is missing); report what is dropped
    extra = np.setdiff1d(ds.ky.values, gs2[c].ky.values)
    if extra.size:
        print(c, lab, "ky not in GS2, dropped:", extra)
        model[(c, lab)] = ds.sel(ky=gs2[c].ky.values)

for c in cases:
    for lab in codes:
        ds = model[(c, lab)]
        assert np.array_equal(ds.ky.values, gs2[c].ky.values), (c, lab, "ky grid differs from GS2")
        ratio = ds.beta.values / gs2[c].beta.values
        assert np.allclose(ratio, ratio[0], rtol=1e-3), (c, lab, "beta is not GS2 beta x constant")
    print(c, "ky", gs2[c].ky.size, "beta", gs2[c].beta.size, "| pyro beta / GS2 beta f =",
          {lab: round(float(model[(c, lab)].beta.values[0] / gs2[c].beta.values[0]), 4) for lab in codes},
          "| modes", {lab: int(model[(c, lab)].mode.size) for lab in codes})

/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0000'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0003'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0004'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0005'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0006'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0007'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0008'. 
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/normalisation.py:944: UserWarning: Reference values 'tref_electron', 'nref_electron', 'bref_B0', 'lref_minor_radius', 'lref_major_radius' are already set for 'pyroscan_base0009'. 
  warnings.warn(


M1_A1_NE TGLF V3 ky not in GS2, dropped: [0.2 0.3]
M1_A1_NE ky 40 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 1.0, 'GFTM w0.6': 1.0, 'GFTM w0.7': 1.0, 'GFTM w0.8': 1.0, 'TGLF default': 1.0, 'TGLF V5': 1.0, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}
R3 ky 42 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 0.9546, 'GFTM w0.6': 0.9546, 'GFTM w0.7': 0.9546, 'GFTM w0.8': 0.9546, 'TGLF default': 0.9546, 'TGLF V5': 0.9546, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}
R4 ky 42 beta 19 | pyro beta / GS2 beta f = {'GFTM w0.4': 0.8776, 'GFTM w0.6': 0.8776, 'GFTM w0.7': 0.8776, 'GFTM w0.8': 0.8776, 'TGLF default': 0.8776, 'TGLF V5': 0.8776, 'TGLF V3': 1.0} | modes {'GFTM w0.4': 2, 'GFTM w0.6': 2, 'GFTM w0.7': 2, 'GFTM w0.8': 2, 'TGLF default': 2, 'TGLF V5': 2, 'TGLF V3': 2}


## Calculate and select

Nearest GS2 ky to each target (actual values printed). GS2 type per plotted (ky, β) cell from GS2's `pyroscan.nc`
through `mode_classification.indicators` + `classify`: the scan varies only ky and β (`p_prime_type` 0), so the base
pyro's species and geometry hold for every cell. ω there is the scan's time average over the last 20%, not the final
time (differences < 0.01 on checked cells; T, χe/χi, De/χe identical). Code type from stored `tearing_parameter` and
`mode_frequency` per mode with the same cutoffs (T and ω only, see header).

In [5]:
ky_index = {c: [int(np.abs(gs2[c].ky.values - k).argmin()) for k in ky_targets] for c in cases}
for c in cases:
    print(c, "ky target -> GS2 grid:", {k: round(float(gs2[c].ky.values[i]), 4) for k, i in zip(ky_targets, ky_index[c])})

_gs2_ind = {}
def gs2_indicators(case, i, j):
    "mode_classification.indicators on one (ky, beta) cell of GS2's pyroscan.nc"
    if case not in _gs2_ind:   # whole scan once: mc.indicators(scan, ds) takes a scan and its samples
        s = mc.attach_legacy_funcs(gs2_pyroscan[case])
        ind = mc.indicators(s, mc.as_samples(s, s.gk_output.data))
        _gs2_ind[case] = {n: ind[n].values.reshape(s.gk_output.data.growth_rate.shape) for n in ("T", "E", "omega", "chi_ratio")}
    return {n: float(v[i, j]) for n, v in _gs2_ind[case].items()}

gs2_label = {(c, i, j): str(mc.classify(gs2_indicators(c, i, j)))
             for c in cases for i in ky_index[c] for j in range(gs2[c].beta.size)
             if gs2[c].growth_rate.values[i, j] > gamma_floor}

def code_type(T, omega):
    "same cutoffs as mode_classification.classify, on the indicators a GFTM/TGLF linear run has"
    w = mc.ION_DIRECTION * omega
    return np.where((w > 0) & (T < mc.T_BAL), "KBM", np.where((w < 0) & (T > mc.T_TEAR), "MTM", "mixed"))

def eig_T(pyroscan, ky=None):
    "FieldLine T of every (ky, beta, mode) from the scan's stored A_par eigenfunction, optionally on a subset of ky"
    p = pyroscan.base_pyro
    p.gk_output = xr.Dataset({"apar": pyroscan.gk_output.data.eigenfunctions.sel(field="apar")})
    T = FieldLine(p).compute_linear_tearing_parameter().transpose("ky", "beta", "mode")
    return (T if ky is None else T.sel(ky=ky)).values

for k, ds in model.items():   # TGLF V3's older scan stores no tearing_parameter: take it from its stored A_par
    if "tearing_parameter" not in ds:
        model[k] = ds.assign(tearing_parameter=(ds.growth_rate.dims, eig_T(model_pyroscan[k], ds.ky.values)))
code_label = {k: code_type(ds.tearing_parameter.values, ds.mode_frequency.values) for k, ds in model.items()}
dominant_mode = {k: ds.growth_rate.fillna(-np.inf).argmax("mode").values for k, ds in model.items()}   # (ky, beta)
for c in cases:
    n = {t: sum(1 for k, v in gs2_label.items() if k[0] == c and v == t) for t in ("KBM", "MTM", "mixed")}
    print(c, "GS2 type counts at plotted ky (gamma>0):", n)

M1_A1_NE ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.1877, 0.3: 0.2892, 0.5: 0.4923}
R3 ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.2, 0.3: 0.3, 0.5: 0.4923}
R4 ky target -> GS2 grid: {0.1: 0.1115, 0.2: 0.2, 0.3: 0.3, 0.5: 0.4923}


M1_A1_NE GS2 type counts at plotted ky (gamma>0): {'KBM': 43, 'MTM': 4, 'mixed': 2}
R3 GS2 type counts at plotted ky (gamma>0): {'KBM': 32, 'MTM': 31, 'mixed': 11}
R4 GS2 type counts at plotted ky (gamma>0): {'KBM': 14, 'MTM': 13, 'mixed': 8}


## Plot

One figure per (case, code): four ky panels, γ vs β (GS2's β number), with TGLF V5 in every figure for comparison.
Lines follow the dominant mode (argmax γ); `show_subdominant` adds the other mode (dashed, open markers).
Markers: ▲ MTM, ■ KBM, ● mixed.

In [7]:
show_subdominant = False   # also draw each code's other mode (faint line, open markers)
comparison = "TGLF V5"     # drawn in every figure next to the code
marker = {"MTM": "^", "KBM": "s", "mixed": "o"}
line_style = {"GFTM": ":", "TGLF": "--"}   # by code family, from the label prefix
# Each figure is drawn for gamma and for omega; dominant mode and markers (gamma > gamma_floor) come from gamma
plot_fields = {"growth_rate": r"$\gamma$ [$c_s/a$]", "mode_frequency": r"$\omega$ [$c_s/a$]"}
figs = {}
for f, c in [(f, c) for f in plot_fields for c in cases]:   # one figure per field
    ref, beta = gs2[c], gs2[c].beta.values
    jb = np.arange(beta.size)
    for lab in codes:
        series = [(lab, "C0")] + ([(comparison, "C1")] if lab != comparison else [])
        fig, axes = plt.subplots(1, len(ky_targets), figsize=(4 * len(ky_targets), 3.6), sharey=True)
        for ax, i in zip(axes, ky_index[c]):
            ax.plot(beta, ref[f].values[i], alpha=0.6, color="k", marker="none")
            for j, g in enumerate(ref[f].values[i]):
                if (c, i, j) in gs2_label:
                    ax.plot(beta[j], g, color="k", marker=marker[gs2_label[(c, i, j)]], ls="")
            for name, col in series:
                g, y, d, typ = model[(c, name)].growth_rate.values[i], model[(c, name)][f].values[i], dominant_mode[(c, name)][i], code_label[(c, name)][i]
                ls = line_style[name.split()[0]]
                ax.plot(beta, y[jb, d], color=col, ls=ls, alpha=0.6, marker="none")
                if show_subdominant:
                    ax.plot(beta, y[jb, 1 - d], color=col, ls=ls, alpha=0.4, marker="none")   # NMODES = 2
                for j, m in np.argwhere(g > gamma_floor):
                    if m == d[j] or show_subdominant:
                        ax.plot(beta[j], y[j, m], color=col, marker=marker[typ[j, m]], ls="", mfc=col if m == d[j] else "none")
            ax.set(xlabel=r"$\beta_e$", title=rf"$k_y$ = {float(ref.ky[i]):.3f}")
        key = [Line2D([], [], color="k", ls="", marker="o", label="GS2"),
               *[Line2D([], [], color=col, ls="", marker="o", label=name) for name, col in series],
               *([Line2D([], [], color="grey", alpha=0.4, label="other mode")] if show_subdominant else []),
               *[Line2D([], [], color="k", ls="", marker=m, label=t if t.isupper() else t.capitalize()) for t, m in marker.items()]]
        axes[0].set_ylabel(plot_fields[f])
        axes[-1].legend(handles=key, loc="best", fontsize=8)
        fig.suptitle(rf"{c}", y=1.04)
        figs[(c, lab, f)] = fig
print(len(figs), "figures")
plt.show()


21 figures


/scratch_local/ipykernel_4158049/707744643.py:11: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, len(ky_targets), figsize=(4 * len(ky_targets), 3.6), sharey=True)
/scratch_local/ipykernel_4158049/707744643.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
ky_comparisons = {"TGLF default": "C1", "TGLF V3": "C2"}   # all drawn in every single-ky figure
ky_codes = [lab for lab in codes if lab != "TGLF V5"]   # TGLF V5 left out of the single-ky figures
legend_name = {"GFTM w0.4": "GFTM"}   # display name in the single-ky legends; other labels unchanged
reference_beta = {
    "M1_A1_NE": 0.036,
    "R3": 0.094,
    "R4": 0.053,
}
figs_ky = {}   # one figure per (case, code, ky, field); same lines and markers as the panel figures above
for f, c in [(f, c) for f in plot_fields for c in cases]:   # one figure per field
    ref, beta = gs2[c], gs2[c].beta.values
    jb = np.arange(beta.size)
    for lab in ky_codes:
        series = ([] if lab in ky_comparisons else [(lab, "C0")]) + list(ky_comparisons.items())
        key = [Line2D([], [], color="k", ls="", marker="o", label="GS2"),
               *[Line2D([], [], color=col, ls="", marker="o", label=legend_name.get(name, name)) for name, col in series],
               *([Line2D([], [], color="grey", alpha=0.4, label="other mode")] if show_subdominant else []),
               *[Line2D([], [], color="k", ls="", marker=m, label=t if t.isupper() else t.capitalize()) for t, m in marker.items()]]
        for i in ky_index[c]:
            fig, ax = plt.subplots(figsize=(5, 3.8))
            ax.plot(beta, ref[f].values[i], alpha=0.6, color="k", marker="none")
            for j, g in enumerate(ref[f].values[i]):
                if (c, i, j) in gs2_label:
                    ax.plot(beta[j], g, alpha=0.6, color="k", marker=marker[gs2_label[(c, i, j)]], ls="")
            for name, col in series:
                g, y, d, typ = model[(c, name)].growth_rate.values[i], model[(c, name)][f].values[i], dominant_mode[(c, name)][i], code_label[(c, name)][i]
                ls = line_style[name.split()[0]]
                ax.plot(beta, y[jb, d], alpha=0.6, color=col, ls=ls, marker="none")
                if show_subdominant:
                    ax.plot(beta, y[jb, 1 - d], color=col, ls=ls, alpha=0.4, marker="none")   # NMODES = 2
                for j, m in np.argwhere(g > gamma_floor):
                    if m == d[j] or show_subdominant:
                        ax.plot(beta[j], y[j, m], alpha=0.6, color=col, marker=marker[typ[j, m]], ls="", mfc=col if m == d[j] else "none")

            ky = float(ref.ky[i])
            ax.set(xlabel=r"$\beta_e$", ylabel=plot_fields[f], title=rf"{c}, $k_y$ = {ky:.3f}")
            ax.axvline(reference_beta[c], color="red", linestyle="--", alpha=0.4, marker="none")
            ax.legend(handles=key, loc="best", fontsize=8)
            figs_ky[(c, lab, ky, f)] = fig
            plt.close(fig)   # too many to display, check the saved PNGs
print(len(figs_ky), "figures")
plt.show()


72 figures


/scratch_local/ipykernel_4158049/2433615849.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from pyrokinetics.pyroscan import PyroScan, PyroScanGKOutput

s13_case = "S1_3"                                    # STEP-EC-HD
s13_ky_targets = [0.1, 0.2, 0.3, 0.5]                # snapped to the nearest GS2 grid value, shown in the panel titles
s13_xlabel = {"q": r"$q$", "shat": r"$\hat{s}$"}
# label -> (code, project suffix: scan_<par>_<suffix>, leaf). Same settings as the beta scans above.
s13_codes = {
    "GFTM w0.4": ("GFTM", "f05", "w0p4"), "GFTM w0.6": ("GFTM", "f05", "w0p6"),
    "GFTM w0.7": ("GFTM", "f05", "w0p7"), "GFTM w0.8": ("GFTM", "f05", "w0p8"),
    "TGLF default": ("TGLF", "modes", "default"), "TGLF V5": ("TGLF", "modes", "v5"), "TGLF V3": ("TGLF", "modes", "v3"),
}


def s13_v(da):
    return np.asarray(getattr(da.data, "magnitude", da.data))


### (10) S1_3 single-ky γ vs q and γ vs ŝ (Fusion_PhD-38e9.18, user request 2026-10-06)

Same format as the single-ky γ vs β figures above (one figure per code and ky; GS2 black, TGLF default C1, TGLF V3 C2, same line styles, `legend_name`), but x = q or ŝ, plain markers (no mode classification, so no classifier is needed), and a red dashed vertical line (`alpha=0.4`, not in the legend, as the β single-kᵧ figures) at the q / ŝ of the GS2 template. The template value is read with pyro from the scan's base pyro (`pyroscan_base.input`) and checked against `GS2/Templates/S1_3`. Data: each leaf's `pyroscan.nc` only; dominant mode = argmax γ.

In [6]:
def s18_template(par):
    "q or shat of the GS2 template: the scan's base pyro, asserted equal to GS2/Templates/S1_3 read with pyro"
    attr = {"q": "q", "shat": "shat"}[par]
    leaf = data_root / "GS2" / run_template / f"scan_{par}" / s13_case / "parameter_scan_GS2"
    base = getattr(Pyro(gk_file=leaf / "pyroscan_base.input", gk_code="GS2").local_geometry, attr)
    tmpl = getattr(Pyro(gk_file=Path(os.environ["GYRO_DATA_INPUT"]) / "GS2" / "Templates" / s13_case / "gs2.in", gk_code="GS2").local_geometry, attr)
    assert np.isclose(float(base), float(tmpl)), (par, base, tmpl)
    print(f"template {par}: scan base pyro {float(base):.6g}, GS2/Templates/{s13_case} {float(tmpl):.6g}")
    return float(base)


def s18_load(par):
    "GS2 and every S1_3 code leaf from pyroscan.nc: growth_rate and mode_frequency on (ky, par), dominant mode = argmax gamma"
    leaf = data_root / "GS2" / run_template / f"scan_{par}" / s13_case / "parameter_scan_GS2"
    ds = PyroScanGKOutput.from_netcdf(leaf / "pyroscan.nc").data
    d = dict(x=s13_v(ds[par]), ky=s13_v(ds.ky), g=s13_v(ds.growth_rate), w=s13_v(ds.mode_frequency), model={}, model_w={}, ref=s18_template(par))
    for lab, (code, suffix, name) in s13_codes.items():
        c = PyroScanGKOutput.from_netcdf(data_root / code / run_template / f"scan_{par}_{suffix}" / s13_case / name / "pyroscan.nc").data
        assert np.allclose(s13_v(c.ky), d["ky"]) and np.allclose(s13_v(c[par]), d["x"]), (lab, "grid differs from GS2")
        g = s13_v(c.growth_rate)
        dom = np.nan_to_num(g, nan=-np.inf).argmax(-1)[..., None]
        d["model"][lab] = np.take_along_axis(g, dom, -1)[..., 0]   # (ky, par)
        d["model_w"][lab] = np.take_along_axis(s13_v(c.mode_frequency), dom, -1)[..., 0]
    return d


s18_fields = [("g", "model", r"$\gamma$ [$c_s/a$]"), ("w", "model_w", r"$\omega$ [$c_s/a$]")]


def s18_plot(par, d):
    sym = {"q": "q", "shat": r"\hat{s}"}[par]
    iky = [int(np.abs(d["ky"] - k).argmin()) for k in s13_ky_targets]
    figs = {}
    for lab in ky_codes:
        series = ([] if lab in ky_comparisons else [(lab, "C0")]) + list(ky_comparisons.items())
        key = [Line2D([], [], color="k", ls="", marker="o", label="GS2"),
               *[Line2D([], [], color=col, ls="", marker="o", label=legend_name.get(name, name)) for name, col in series]]
        for (fk, mk, ylabel), i in [(fl, i) for fl in s18_fields for i in iky]:   # gamma and omega figures
            fig, ax = plt.subplots(figsize=(5, 3.8))
            for g, y, col, ls in [(d["g"][i], d[fk][i], "k", "-")] + [(d["model"][n][i], d[mk][n][i], c, line_style[n.split()[0]]) for n, c in series]:
                ax.plot(d["x"], y, alpha=0.6, color=col, ls=ls, marker="none")
                pos = g > gamma_floor   # markers where gamma > 0, as in the gamma figures
                ax.plot(d["x"][pos], y[pos], alpha=0.6, color=col, marker="o", ls="")
            ax.axvline(d["ref"], color="red", linestyle="--", alpha=0.4, marker="none")
            ax.set(xlabel=s13_xlabel[par], ylabel=ylabel, title=rf"{s13_case}, $k_y$ = {d['ky'][i]:.3f}")
            ax.legend(handles=key, loc="best", fontsize=8)
            figs[(lab, float(d["ky"][i]), fk)] = fig
            plt.close(fig)
    return figs


output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
for par in ("q", "shat"):
    figs18 = s18_plot(par, s18_load(par))
    for (lab, ky, fk), fig in figs18.items():
        fig.savefig(output_dir / f"{lab.replace(' ', '_').replace('.', 'p').lower()}_gs2_{'omega' if fk == 'w' else 'gamma'}_vs_{par}_{s13_case}_ky{ky:.3f}.png", dpi=150, bbox_inches="tight")
    print(par, "saved", len(figs18), "single-ky figures to", output_dir)

/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/pyro.py:1792: UserWarning: Explicitly set value of beta (0.09 beta_ref_ee_B0_pyroscan_base0002) is inconsistent with value from physical reference values (0.09609377161346086 beta_ref_ee_B0_pyroscan_base0002)
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/pyro.py:1792: UserWarning: Explicitly set value of beta (0.09 beta_ref_ee_B0_gs20000) is inconsistent with value from physical reference values (0.09609377161346086 beta_ref_ee_B0_gs20000)
  warnings.warn(


template q: scan base pyro 3.49332, GS2/Templates/S1_3 3.49332


q saved 24 single-ky figures to /pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/Plots/beta_stabilisation_f05


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/pyro.py:1792: UserWarning: Explicitly set value of beta (0.09 beta_ref_ee_B0_pyroscan_base0003) is inconsistent with value from physical reference values (0.09609377161346086 beta_ref_ee_B0_pyroscan_base0003)
  warnings.warn(


/pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/.venv/lib/python3.14/site-packages/pyrokinetics/pyro.py:1792: UserWarning: Explicitly set value of beta (0.09 beta_ref_ee_B0_gs20001) is inconsistent with value from physical reference values (0.09609377161346086 beta_ref_ee_B0_gs20001)
  warnings.warn(


template shat: scan base pyro 1.20498, GS2/Templates/S1_3 1.20498


shat saved 24 single-ky figures to /pitagora_work/FUPB2_QLTURB/fwatts/Fusenet_Poster/General_Analysis/Plots/beta_stabilisation_f05
